In [1]:
import sqlite3
import pandas as pd

In [2]:
%load_ext sql
%sql sqlite:///rappi_sql_practice.db

Exploracion inicial

In [3]:
%%sql
SELECT name
FROM sqlite_master
WHERE type = 'table';

 * sqlite:///rappi_sql_practice.db
Done.


name
cities
customers
restaurants
couriers
products
orders
order_items
payments
ratings


In [20]:
%%sql
PRAGMA table_info(restaurants);

 * sqlite:///rappi_sql_practice.db
Done.


cid,name,type,notnull,dflt_value,pk
0,restaurant_id,INTEGER,0,None,1
1,restaurant_name,TEXT,1,None,0
2,city_id,INTEGER,1,None,0
3,category,TEXT,1,None,0
4,is_active,INTEGER,1,1,0
5,opened_date,DATE,1,None,0


Pregunta 1

Para cada status:

número total de órdenes

ordenado de mayor a menor

In [5]:
%%sql
SELECT status, count(*) as c
FROM orders
GROUP BY status
ORDER BY c DESC;


 * sqlite:///rappi_sql_practice.db
Done.


status,c
delivered,15
picked_up,14
assigned,10
paid,9
created,6
cancelled,6


Número de órdenes por día
(usa date(order_ts))

Debe mostrar:

date

total_orders

ordenado del día con más órdenes al de menos

In [8]:
%%sql
SELECT date(order_ts) AS order_date, COUNT(*) AS total_orders
FROM orders
GROUP BY date(order_ts)
ORDER BY total_orders DESC
LIMIT 5;

 * sqlite:///rappi_sql_practice.db
Done.


order_date,total_orders
2025-12-25,4
2026-02-18,3
2025-12-22,3
2026-02-16,2
2026-02-03,2


aca usamos el date para pues solo coger las fechas y que no sea el timestamp completo

¿Qué porcentaje de órdenes están canceladas?

Debe devolver:

cancel_rate (en porcentaje)

In [14]:
%%sql
SELECT COUNT(*) * 1.0 /(SELECT COUNT(*) FROM orders) * 100 AS cancel_rate
FROM orders
WHERE status = 'cancelled';



 * sqlite:///rappi_sql_practice.db
Done.


cancel_rate
10.0


In [15]:
%%sql
SELECT 
    SUM(CASE WHEN status = 'cancelled' THEN 1 ELSE 0 END) * 1.0
    / COUNT(*) * 100 AS cancel_rate
FROM orders;

 * sqlite:///rappi_sql_practice.db
Done.


cancel_rate
10.0


Dos maneras de hacer esto

Pregunta 3
Ticket promedio por restaurante
Incluye:

restaurant_name

avg_ticket

ordenado de mayor a menor

Aquí necesitas JOIN + GROUP BY.

In [18]:
%%sql
SELECT * FROM payments
LIMIT 5;


 * sqlite:///rappi_sql_practice.db
Done.


payment_id,order_id,method,amount,paid_ts
1,1,cash,111.0,None
2,2,cash,46.9,None
3,3,card,204.2,2026-02-08 20:24:00
4,4,card,14.0,2026-01-15 07:10:00
5,5,wallet,83.5,2026-02-13 11:28:00


In [24]:
%%sql
SELECT * FROM restaurants
LIMIT 5;

 * sqlite:///rappi_sql_practice.db
Done.


restaurant_id,restaurant_name,city_id,category,is_active,opened_date
1,Arepa & Co,1,colombian,1,2022-08-15
2,Sushi Nori,1,japanese,1,2023-03-20
3,Pizza Porto,2,italian,1,2021-11-02
4,Green Bowl,2,healthy,1,2024-01-10
5,Taco Loco,3,mexican,1,2020-06-05


In [29]:
%%sql
SELECT AVG(amount) as avg_ticket
FROM payments 

 * sqlite:///rappi_sql_practice.db
Done.


avg_ticket
79.02833333333335


In [25]:
%%sql
SELECT AVG(p.amount) as avg_ticket, r.restaurant_name
FROM payments p
JOIN orders o ON p.order_id = o.order_id
JOIN restaurants r ON o.restaurant_id = r.restaurant_id
GROUP BY r.restaurant_name
ORDER BY avg_ticket DESC;


 * sqlite:///rappi_sql_practice.db
Done.


avg_ticket,restaurant_name
139.9375,Sushi Nori
92.53636363636362,Pizza Porto
72.7,Green Bowl
71.47142857142856,Arepa & Co
61.5,Taco Loco
46.0,Burger Lab


Aca empezamos definiendo lo que queremos que en este caso era el amount que estaba en la tabla payments esto lo relacionamos con la tabla orders a tarves d elas llaves fornaeas usando el join y ON para poner la condicion despues hacemos lo mismo pero ahora con restaurants y terminamos agrupando esto por el nombre y ordenandolo por el avg ticket

Pregunta 4

Revenue total por restaurante.

Recuerda:

No es promedio

Es suma total

In [36]:
%%sql
SELECT SUM(p.amount) as total_amount, r.restaurant_name
FROM payments p
JOIN orders o ON p.order_id = o.order_id
JOIN restaurants r ON o.restaurant_id = r.restaurant_id
GROUP BY r.restaurant_name
ORDER BY total_amount DESC;

 * sqlite:///rappi_sql_practice.db
Done.


total_amount,restaurant_name
1119.5,Sushi Nori
1090.5,Green Bowl
1017.8999999999999,Pizza Porto
553.5,Taco Loco
500.29999999999995,Arepa & Co
460.0,Burger Lab


In [40]:
%%sql
SELECT * FROM payments p
WHERE p.paid_ts IS NULL;

 * sqlite:///rappi_sql_practice.db
Done.


payment_id,order_id,method,amount,paid_ts
1,1,cash,111.0,None
2,2,cash,46.9,None
14,14,cash,107.0,None
15,15,card,126.0,None
23,23,cash,56.0,None
29,29,pse,62.5,None
36,36,cash,110.0,None
42,42,wallet,73.0,None
51,51,cash,33.0,None
52,52,pse,24.5,None


In [37]:
%%sql
SELECT SUM(p.amount) as total_amount, r.restaurant_name
FROM payments p
JOIN orders o ON p.order_id = o.order_id
JOIN restaurants r ON o.restaurant_id = r.restaurant_id
WHERE p.paid_ts IS NOT NULL
GROUP BY r.restaurant_name
ORDER BY total_amount DESC;

 * sqlite:///rappi_sql_practice.db
Done.


total_amount,restaurant_name
946.5,Green Bowl
927.5,Sushi Nori
882.3999999999999,Pizza Porto
428.9,Arepa & Co
371.0,Burger Lab
336.5,Taco Loco


Pregunta 5
Para cada restaurante calcula:

total_orders (solo pagadas)

total_revenue

avg_ticket

In [42]:
%%sql
SELECT SUM(p.amount) as total_amount, AVG(p.amount) as avg_amount, count(*) as order_count, r.restaurant_name
FROM payments p
JOIN orders o ON p.order_id = o.order_id
JOIN restaurants r ON o.restaurant_id = r.restaurant_id
WHERE p.paid_ts IS NOT NULL
GROUP BY r.restaurant_name
ORDER BY total_amount DESC;

 * sqlite:///rappi_sql_practice.db
Done.


total_amount,avg_amount,order_count,restaurant_name
946.5,72.8076923076923,13,Green Bowl
927.5,154.58333333333334,6,Sushi Nori
882.3999999999999,98.04444444444442,9,Pizza Porto
428.9,85.78,5,Arepa & Co
371.0,46.375,8,Burger Lab
336.5,48.07142857142857,7,Taco Loco


Pregunta 6
Calcula el success_rate por restaurante.
success_rate = delivered_orders / total_orders
Incluye:

restaurant_name

total_orders

delivered_orders

success_rate (en porcentaje)

Aquí necesitarás CASE WHEN.

In [67]:
%%sql
SELECT
r.restaurant_name,
COUNT(*) AS total_orders,
SUM(CASE WHEN o.status = 'delivered' THEN 1 ELSE 0 END) AS delivered_orders,
100.0 * SUM(CASE WHEN o.status = 'delivered' THEN 1 ELSE 0 END) / COUNT(*) AS success_rate_pct
FROM orders o
JOIN restaurants r
ON o.restaurant_id = r.restaurant_id

JOIN payments p
ON p.order_id = o.order_id
WHERE p.paid_ts IS NOT NULL

GROUP BY r.restaurant_name
ORDER BY success_rate_pct DESC;

 * sqlite:///rappi_sql_practice.db
Done.


restaurant_name,total_orders,delivered_orders,success_rate_pct
Pizza Porto,9,5,55.55555555555556
Arepa & Co,5,2,40.0
Sushi Nori,6,2,33.333333333333336
Taco Loco,7,2,28.571428571428573
Green Bowl,13,3,23.076923076923077
Burger Lab,8,1,12.5


Pregunta 7
Top 3 clientes por gasto total (solo pagadas), mostrando nombre y total_spent

In [68]:
%%sql
SELECT * FROM customers;

 * sqlite:///rappi_sql_practice.db
Done.


customer_id,full_name,email,city_id,signup_date,is_premium
1,Sofía Zarruk,sofia@example.com,1,2024-08-15,1
2,Juan Pérez,juan@example.com,1,2025-05-07,0
3,María Gómez,maria@example.com,2,2025-09-24,0
4,Camila Rojas,camila@example.com,2,2026-02-20,1
5,Andrés Silva,andres@example.com,3,2024-02-16,0
6,Valentina Torres,vale@example.com,4,2025-09-20,0
7,Diego Castro,diego@example.com,3,2025-12-26,1


In [70]:
%%sql
SELECT * FROM payments
LIMIT 5;

 * sqlite:///rappi_sql_practice.db
Done.


payment_id,order_id,method,amount,paid_ts
1,1,cash,111.0,None
2,2,cash,46.9,None
3,3,card,204.2,2026-02-08 20:24:00
4,4,card,14.0,2026-01-15 07:10:00
5,5,wallet,83.5,2026-02-13 11:28:00


In [71]:
%%sql
SELECT * FROM orders
LIMIT 5;

 * sqlite:///rappi_sql_practice.db
Done.


order_id,customer_id,restaurant_id,courier_id,order_ts,status,delivery_fee,promo_code
1,3,4,None,2026-01-06 20:57:00,created,6.0,None
2,1,1,1,2026-02-14 20:23:00,cancelled,7.0,NUEVO15
3,3,3,4,2026-02-08 20:24:00,picked_up,7.0,None
4,5,5,5,2026-01-15 07:10:00,picked_up,5.0,RAPPI10
5,5,5,5,2026-02-13 11:28:00,delivered,4.5,NUEVO15


In [73]:
%%sql
SELECT c.full_name, SUM(p.amount) as total_spent
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN payments p ON o.order_id = p.order_id
WHERE p.paid_ts IS NOT NULL
GROUP BY c.full_name
ORDER BY total_spent DESC;


 * sqlite:///rappi_sql_practice.db
Done.


full_name,total_spent
Juan Pérez,862.7
Camila Rojas,854.8999999999999
María Gómez,751.6
Sofía Zarruk,566.2
Valentina Torres,380.9
Andrés Silva,246.5
Diego Castro,230.0


pregunta 8
¿Cuál es la hora pico del día?

In [89]:
%%sql
SELECT strftime('%H', order_ts) as hora, count(*) as ordenes_por_hora
FROM orders
GROUP BY hora
ORDER BY ordenes_por_hora DESC;

 * sqlite:///rappi_sql_practice.db
Done.


hora,ordenes_por_hora
20,5
18,5
05,5
03,5
11,4
07,4
06,4
00,4
23,3
16,3


Pregunta 10 ¿Cuál es la ciudad con mayor revenue total?

In [90]:
%%sql
SELECT * FROM cities;

 * sqlite:///rappi_sql_practice.db
Done.


city_id,name
1,Bogotá
2,Medellín
3,Cali
4,Barranquilla


In [99]:
%%sql
SELECT * FROM orders
LIMIT 5;

 * sqlite:///rappi_sql_practice.db
Done.


order_id,customer_id,restaurant_id,courier_id,order_ts,status,delivery_fee,promo_code
1,3,4,None,2026-01-06 20:57:00,created,6.0,None
2,1,1,1,2026-02-14 20:23:00,cancelled,7.0,NUEVO15
3,3,3,4,2026-02-08 20:24:00,picked_up,7.0,None
4,5,5,5,2026-01-15 07:10:00,picked_up,5.0,RAPPI10
5,5,5,5,2026-02-13 11:28:00,delivered,4.5,NUEVO15


In [95]:
%%sql
SELECT * FROM customers
LIMIT 5;

 * sqlite:///rappi_sql_practice.db
Done.


customer_id,full_name,email,city_id,signup_date,is_premium
1,Sofía Zarruk,sofia@example.com,1,2024-08-15,1
2,Juan Pérez,juan@example.com,1,2025-05-07,0
3,María Gómez,maria@example.com,2,2025-09-24,0
4,Camila Rojas,camila@example.com,2,2026-02-20,1
5,Andrés Silva,andres@example.com,3,2024-02-16,0


In [104]:
%%sql
SELECT ci.name, SUM(p.amount) as total_revenue
FROM payments p
JOIN orders o ON p.order_id = o.order_id
JOIN customers c ON o.customer_id = c.customer_id
JOIN cities ci ON c.city_id = ci.city_id
WHERE p.paid_ts IS NOT NULL
GROUP BY ci.name;


 * sqlite:///rappi_sql_practice.db
Done.


name,total_revenue
Barranquilla,380.9
Bogotá,1428.9
Cali,476.5
Medellín,1606.4999999999998
